### Improting the libs

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, dense_rank, col, sum
from pyspark.sql.types import StructField, StructType, StringType, IntegerType
from pyspark.sql.window import Window

### Creating spark val

In [3]:
# defining spark val
spark = SparkSession.Builder().master("local[5]").appName("testing").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/18 10:25:42 WARN Utils: Your hostname, Vishvas-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.113 instead (on interface en0)
26/06/18 10:25:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/18 10:25:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark

### defining schema and reading the files

In [5]:
# create schema using structType and structField
# employee_id,employee_name,department,salary,age,city
schema = StructType(
    [
        StructField("employee_id", IntegerType(), False),
        StructField("employee_name", StringType(), False),
        StructField("department", StringType(), False),
        StructField("salary", IntegerType(), False),
        StructField("age", IntegerType(), False),
        StructField("city", StringType(), False)
    ]
)

In [6]:
df = spark.read.format("csv").option("inferSchmea", "true").option("header", "true").schema(schema).load("data.csv")

In [7]:
df.show()

+-----------+-------------+-----------+------+---+---------+
|employee_id|employee_name| department|salary|age|     city|
+-----------+-------------+-----------+------+---+---------+
|          1|   Employee_1|Engineering| 36500| 23|   Mumbai|
|          2|   Employee_2|         HR| 38000| 24|Bengaluru|
|          3|   Employee_3|    Finance| 39500| 25|     Pune|
|          4|   Employee_4|      Sales| 41000| 26|Hyderabad|
|          5|   Employee_5|  Marketing| 41000| 27|    Delhi|
|          6|   Employee_6| Operations| 41000| 28|   Mumbai|
|          7|   Employee_7|Engineering| 41000| 29|Bengaluru|
|          8|   Employee_8|         HR| 41000| 30|     Pune|
|          9|   Employee_9|    Finance| 41000| 31|Hyderabad|
|         10|  Employee_10|      Sales| 41000| 32|    Delhi|
|         11|  Employee_11|  Marketing| 51500| 33|   Mumbai|
|         12|  Employee_12| Operations| 53000| 34|Bengaluru|
|         13|  Employee_13|Engineering| 54500| 35|     Pune|
|         14|  Employee_

### find the highest sal from each department

In [ ]:
windowSpecs = Window.partitionBy(col("department")).orderBy(col("salary").desc())
highestSal = df.withColumn("rank", dense_rank().over(windowSpecs)).select("employee_id","employee_name","department","salary","age","city").filter(col("rank")==1)
highestSal.show()

+-----------+-------------+-----------+------+---+---------+
|employee_id|employee_name| department|salary|age|     city|
+-----------+-------------+-----------+------+---+---------+
|         49|  Employee_49|Engineering|108500| 31|Hyderabad|
|         45|  Employee_45|    Finance|102500| 27|    Delhi|
|         50|  Employee_50|         HR|110000| 32|    Delhi|
|         47|  Employee_47|  Marketing|105500| 29|Bengaluru|
|         48|  Employee_48| Operations|107000| 30|     Pune|
|         46|  Employee_46|      Sales|104000| 28|   Mumbai|
+-----------+-------------+-----------+------+---+---------+



26/06/18 11:26:26 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 930367 ms exceeds timeout 120000 ms
26/06/18 11:26:26 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/18 11:26:28 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

### calculate the running sum of salary order by emp id

In [21]:
windowSpecs = Window.orderBy(col("employee_id").asc())
df.withColumn("runningSum", sum(col("salary")).over(windowSpecs)).select("employee_id","employee_name","salary","runningSum").show(100, False)

+-----------+-------------+------+----------+
|employee_id|employee_name|salary|runningSum|
+-----------+-------------+------+----------+
|1          |Employee_1   |36500 |36500     |
|2          |Employee_2   |38000 |74500     |
|3          |Employee_3   |39500 |114000    |
|4          |Employee_4   |41000 |155000    |
|5          |Employee_5   |41000 |196000    |
|6          |Employee_6   |41000 |237000    |
|7          |Employee_7   |41000 |278000    |
|8          |Employee_8   |41000 |319000    |
|9          |Employee_9   |41000 |360000    |
|10         |Employee_10  |41000 |401000    |
|11         |Employee_11  |51500 |452500    |
|12         |Employee_12  |53000 |505500    |
|13         |Employee_13  |54500 |560000    |
|14         |Employee_14  |56000 |616000    |
|15         |Employee_15  |57500 |673500    |
|16         |Employee_16  |59000 |732500    |
|17         |Employee_17  |60500 |793000    |
|18         |Employee_18  |62000 |855000    |
|19         |Employee_19  |63500 |

In [25]:
# running sum based on salary in asc
windowSpecs = Window.orderBy(col("salary").asc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)
df.withColumn("runningSum", sum(col("salary")).over(windowSpecs)).select("employee_id","employee_name","salary","runningSum").show(100, False)

+-----------+-------------+------+----------+
|employee_id|employee_name|salary|runningSum|
+-----------+-------------+------+----------+
|1          |Employee_1   |36500 |36500     |
|2          |Employee_2   |38000 |74500     |
|3          |Employee_3   |39500 |114000    |
|4          |Employee_4   |41000 |155000    |
|5          |Employee_5   |41000 |196000    |
|6          |Employee_6   |41000 |237000    |
|7          |Employee_7   |41000 |278000    |
|8          |Employee_8   |41000 |319000    |
|9          |Employee_9   |41000 |360000    |
|10         |Employee_10  |41000 |401000    |
|11         |Employee_11  |51500 |452500    |
|12         |Employee_12  |53000 |505500    |
|13         |Employee_13  |54500 |560000    |
|14         |Employee_14  |56000 |616000    |
|15         |Employee_15  |57500 |673500    |
|16         |Employee_16  |59000 |732500    |
|17         |Employee_17  |60500 |793000    |
|18         |Employee_18  |62000 |855000    |
|19         |Employee_19  |63500 |

PySpark **Window functions** are used to calculate results across a range of input rows associated with a specific row. Unlike regular aggregate functions (which collapse rows into a single result), Window functions allow you to compute aggregates, rankings, or analytical values **while preserving the original row identity**.

Here is a comprehensive cheat sheet and guide on how to master them.

---

### 1. The Three Core Components

To use a window function, you must first define a **Window Specification** using `Window.partitionBy()`, `orderBy()`, and optionally a frame specification.

```python
from pyspark.sql.window import Window
from pyspark.sql import functions as F

windowSpec = Window.partitionBy("department").orderBy("salary")

```

* **`partitionBy`**: Defines how the rows are grouped (similar to `GROUP BY`). If omitted, the entire dataset is treated as one giant partition.
* **`orderBy`**: Defines the sequence of rows within each partition. **Crucial for ranking and analytical functions.**
* **Frame Specification (Optional)**: Defines the boundaries of the window subset relative to the current row (e.g., "take the last 3 rows up to the current row").

---

### 2. Types of Window Functions

Window functions generally fall into three buckets: **Ranking**, **Analytic**, and **Aggregate**.

#### A. Ranking Functions

These functions assign a mathematical rank to rows within a partition based on the `orderBy` clause.

| Function | Description | Example Output (for identical values: 100, 100, 200) |
| --- | --- | --- |
| `row_number()` | Assigns a unique, sequential number starting at 1. | `1, 2, 3` |
| `rank()` | Assigns ranks with gaps if there are duplicate values. | `1, 1, 3` |
| `dense_rank()` | Assigns ranks *without* gaps for duplicate values. | `1, 1, 2` |
| `percent_rank()` | Computes the relative rank of a row (percentile). | `0.0, 0.0, 1.0` |
| `ntile(n)` | Divides the rows into `n` roughly equal groups. | If $n=2$: `1, 1, 2` |

```python
# Code Example
df.withColumn("row_num", F.row_number().over(windowSpec)) \
  .withColumn("rank", F.rank().over(windowSpec)) \
  .withColumn("dense_rank", F.dense_rank().over(windowSpec))

```

#### B. Analytic / Value Functions

These functions allow you to peer forward or backward in your dataset without performing self-joins.

* **`lag(col, offset=1, default=None)`**: Grabs a value from `offset` rows *before* the current row. Great for calculating year-over-year growth.
* **`lead(col, offset=1, default=None)`**: Grabs a value from `offset` rows *after* the current row.
* **`first(col)` / `last(col)**`: Returns the first or last value in the window frame.

```python
# Code Example: Get the previous employee's salary in the same department
windowDept = Window.partitionBy("department").orderBy("salary")

df.withColumn("previous_salary", F.lag("salary", 1).over(windowDept)) \
  .withColumn("next_salary", F.lead("salary", 1).over(windowDept))

```

#### C. Aggregate Functions

Any standard PySpark aggregate function (`sum`, `avg`, `min`, `max`, `count`) can be turned into a window function by appending `.over()`.

```python
# Running Total Example
windowRunningTotal = Window.partitionBy("department").orderBy("date")

df.withColumn("running_total", F.sum("sales").over(windowRunningTotal)) \
  .withColumn("dept_average", F.avg("sales").over(Window.partitionBy("department"))) # No order by = total dept average on every row

```

---

### 3. Advanced: Window Frames (`rowsBetween` vs `rangeBetween`)

When you use an `orderBy` clause with an aggregate function, PySpark defaults to a moving frame (`Window.unboundedPreceding` to `Window.currentRow`). If you want to customize this boundary, you use a Frame Specification.

* **`rowsBetween(start, end)`**: Based on physical row positions relative to the current row.
* **`rangeBetween(start, end)`**: Based on the *values* of the column in the `orderBy` clause.

**Special Constants:**

* `Window.unboundedPreceding` ($-\infty$)
* `Window.unboundedFollowing` ($+\infty$)
* `Window.currentRow` (0)

```python
# Example: 3-row moving average (1 row before, current row, 1 row after)
moving_window = Window.partitionBy("department").orderBy("date").rowsBetween(-1, 1)

df.withColumn("rolling_avg", F.avg("sales").over(moving_window))

```

---

### 4. Complete Code Implementation

Here is a quick, copy-pasteable example to see it all in action:

```python
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("WindowDemo").getOrCreate()

# Sample Data
data = [
    ("Sales", "Alice", 5000), ("Sales", "Bob", 7000), ("Sales", "Charlie", 5000),
    ("Marketing", "David", 6000), ("Marketing", "Eve", 8000)
]
columns = ["Department", "Name", "Salary"]
df = spark.createDataFrame(data, columns)

# Define Window
windowSpec = Window.partitionBy("Department").orderBy(F.desc("Salary"))

# Apply Window Functions
result_df = df.withColumn("Rank", F.rank().over(windowSpec)) \
              .withColumn("DenseRank", F.dense_rank().over(windowSpec)) \
              .withColumn("RowNumber", F.row_number().over(windowSpec))

result_df.show()

```

#### Output:

```text
+----------+-------+------+----+---------+---------+
|Department|   Name|Salary|Rank|DenseRank|RowNumber|
+----------+-------+------+----+---------+---------+
|     Sales|    Bob|  7000|   1|        1|        1|
|     Sales|  Alice|  5000|   2|        2|        2|
|     Sales|Charlie|  5000|   2|        2|        3|
| Marketing|    Eve|  8000|   1|        1|        1|
| Marketing|  David|  6000|   2|        2|        2|
+----------+-------+------+----+---------+---------+

```

---

### 5. Performance & Gotchas ⚠️

1. **The Single Partition Danger**: If you call `Window.orderBy("something")` *without* a `partitionBy()`, PySpark is forced to move **all your data into a single partition on a single executor** to sort it. This will frequently cause Out Of Memory (OOM) errors on large datasets.
2. **`rank()` vs `dense_rank()` with limits**: If you are trying to find the "Top N" rows per group, filter using `row_number()` or `dense_rank()`. If you use `rank()`, you might accidentally skip numbers or grab more rows than intended due to duplicate ties.
3. **Memory Overhead**: Window functions cache data in memory to compute frames. Be mindful of large partition sizes.